# NB11 — RQ2 U_real Segmentation, Dedup, Freeze U_clean

Turns NB10 `ELIGIBLE_SOURCE` + technical `PASS` WAVs into a frozen, de-duplicated,
contamination-protected set of speech segments (`U_clean`).

**Guarantees**
- Consumes only NB10 `ELIGIBLE_SOURCE` + `technical PASS`; source WAV hashes re-verified.
- Sample-exact, **non-overlapping** segments; `duration_seconds = (end_sample - start_sample) / 16000`.
- Deterministic `segment_uid = SHA256(source_id + source_pcm16_sha256 + start_sample + end_sample + segmentation_contract_sha256)`.
- Two dedup layers: exact PCM SHA-256, then perceptual (Chromaprint/fpcalc).
- Overlap protection vs **all three RQ1 final splits** G_train / G_validation / frozen G_test — **audio identity only**.
  No `text_bahnar` / `text_vi` / `text_en`, no reference/prediction/metric, no model, no BLEU/CER/WER, no threshold tuning on test.
- A retained duplicate or retained protected-overlap makes validation **fail closed**.

**Code lives in `src/`** (`rq2_segmentation.py`, `rq2_audio_fingerprint.py`, `rq2_u_clean.py`); this notebook only orchestrates.

**Freeze flags:** segmentation and overlap are frozen, and the full pipeline is enabled:
`RUN_FULL_PIPELINE = True`, `SEGMENTATION_CONFIG_FROZEN = True`, `OVERLAP_CONFIG_FROZEN = True`.
This notebook's frozen OverlapConfig is authoritative. `similarity_threshold=0.775236`
was calibrated on controlled real VOV4 transformations and negative pairs, with `min_overlap_items = 20`.

In [ ]:
import sys
from pathlib import Path

# Make src/ importable from the notebook.
def _find_root(start: Path) -> Path:
    for cand in [start, *start.parents]:
        if (cand / "requirements.txt").is_file() and (cand / "src").is_dir() and (cand / "notebooks").is_dir():
            return cand
    raise RuntimeError("cannot locate project root")

PROJECT_ROOT = _find_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.rq2_segmentation import SegmentationConfig, has_webrtcvad
from src.rq2_audio_fingerprint import OverlapConfig, has_fpcalc
from src import rq2_u_clean as uc

# ---- Freeze / run flags ----
RUN_FULL_PIPELINE = True
SEGMENTATION_CONFIG_FROZEN = True
OVERLAP_CONFIG_FROZEN = True
# Canonical manifest locators are resolved in the config cell. The derived
# protected identity index is built only when RUN_FULL_PIPELINE is True.
REFERENCE_INDEX = None
AUDIO_IDENTITY_INDEX = None
AUDIO_IDENTITY_SHA256 = None

print("PROJECT_ROOT:", PROJECT_ROOT)
print("webrtcvad available:", has_webrtcvad(), "| fpcalc available:", has_fpcalc())
print("RUN_FULL_PIPELINE:", RUN_FULL_PIPELINE)
print("SEGMENTATION_CONFIG_FROZEN:", SEGMENTATION_CONFIG_FROZEN, "| OVERLAP_CONFIG_FROZEN:", OVERLAP_CONFIG_FROZEN)

## 1. Config

Segmentation and overlap are frozen. `similarity_threshold` is `0.775236` and
`min_overlap_items` is 20. The full pipeline is enabled.

In [ ]:
segmentation = SegmentationConfig(
    vad_frame_ms=30,
    vad_mode=2,
    min_segment_seconds=3.0,
    target_segment_seconds=25.0,
    max_segment_seconds=40.0,
    merge_silence_gap_seconds=0.4,
    boundary_search_seconds=2.0,
    min_speech_fraction=0.60,
    energy_fallback=True,
    sample_rate=16000,
    channels=1,
    sample_width_bytes=2,
    frozen=SEGMENTATION_CONFIG_FROZEN,
)
overlap = OverlapConfig(
    fingerprint_length_seconds=120,
    seconds_per_fingerprint_item=0.1238,
    shingle_k=8,
    min_shared_shingles=1,
    min_overlap_items=20,
    similarity_threshold=0.775236,
    frozen=OVERLAP_CONFIG_FROZEN,
)

REFERENCE_INDEX = uc.canonical_reference_index(PROJECT_ROOT)
AUDIO_IDENTITY_INDEX = None
AUDIO_IDENTITY_SHA256 = None
resolver = None
if RUN_FULL_PIPELINE:
    identity = uc.ensure_derived_protected_identity_index(
        project_root=PROJECT_ROOT,
        reference_index=REFERENCE_INDEX,
    )
    AUDIO_IDENTITY_INDEX = identity["relative_path"]
    AUDIO_IDENTITY_SHA256 = identity["sha256"]
    resolver = uc.build_durable_protected_resolver(
        project_root=PROJECT_ROOT,
        reference_index=REFERENCE_INDEX,
        mapping=uc.default_protected_manifest_mapping(),
        audio_identity_index=PROJECT_ROOT / AUDIO_IDENTITY_INDEX,
        audio_identity_sha256=AUDIO_IDENTITY_SHA256,
    )

config = uc.build_config(
    project_root=PROJECT_ROOT,
    run_full_pipeline=RUN_FULL_PIPELINE,
    segmentation=segmentation,
    overlap=overlap,
    protected_reference_resolver=resolver,
)
print("segmentation_contract_sha256:", config.segmentation_contract_sha())
print("overlap_contract_sha256:", config.overlap_contract_sha())

## 2. Orchestration (guarded)

The full pipeline runs when `RUN_FULL_PIPELINE` is True. Segmentation and overlap are frozen.
Exact protected PCM identity runs for every materialized reference before perceptual matching.
U segments and protected clips under 5.1 seconds are outside fpcalc's 20-item range.
A U segment that short is excluded as `EXCLUDED_PERCEPTUAL_INELIGIBLE` after exact PCM dedup and is not fingerprinted.
A protected clip that short skips fpcalc only after its exact PCM check is recorded.

In [ ]:
def run_pipeline(config):
    """Full RQ2 U_clean orchestration. Runs only when RUN_FULL_PIPELINE is True.

    Sequence: lock NB10 + resolve protected references -> resume completed sources
    from sharded checkpoint state -> segment remaining sources -> exact dedup ->
    U perceptual eligibility -> write or reuse verified eligible segment WAVs ->
    fingerprint eligible U segments -> protected reference fingerprints ->
    exact protected PCM -> perceptual U-U dedup + batched protected-overlap
    exclusion -> fail-closed validation -> artifacts.
    """
    uc.require_full_run_dependencies()
    context = uc.validate_inputs(config)  # full run: NB10 locked + 3 splits resolved
    if config.protected_reference_resolver is None:
        raise RuntimeError("full run requires a protected_reference_resolver")

    from src.rq2_segmentation import compute_speech_flags
    from src.rq2_audio_fingerprint import compute_fingerprint

    import time
    state_path = Path(config.out_dir) / "checkpoint" / "state.json"
    cached = uc.load_resumable_checkpoint(context, load_protected_fingerprints=False) if state_path.is_file() else None
    source_status = dict((cached or {}).get("source_status") or {})
    cached_segments = list((cached or {}).get("segments") or [])

    # 1) Segment every eligible source. A completed source stays completed even
    # when it produced zero usable segments.
    segments = []
    for source_row in context.eligible_sources:
        sid = str(source_row["source_id"])
        uc.verify_source_hashes(source_row, config.project_root)
        if source_status.get(sid, {}).get("segmentation_status") == "COMPLETE":
            segments.extend(row for row in cached_segments if str(row.get("source_id")) == sid)
            continue
        info = uc.read_wav_pcm16(uc.resolve_project_path(source_row["wav_local_path"], config.project_root))
        flags = compute_speech_flags(info["pcm"], config.segmentation)
        built = uc.build_segments_for_source(source_row, info["pcm"], flags, config)
        segments.extend(built)
        uc.mark_source_segmentation_complete(context, sid, built)
        source_status[sid] = {"segmentation_status": "COMPLETE", "n_segments": len(built)}

    # 2) Exact PCM dedup, then exclude U segments below the 5.1s fpcalc boundary.
    # Eligible WAVs are written or hash-checked. Ineligible rows are not written.
    uc.exact_deduplicate(segments)
    cached_fps = uc.migrate_cached_u_fingerprints(
        segments, context, (cached or {}).get("segment_fingerprints_by_uid") or {},
    )
    for row in segments:
        if row["segment_uid"] in cached_fps and row["u_clean_status"] == uc.RETAINED_STATUS:
            abs_path = Path(config.out_dir) / "segments" / str(row["source_id"]) / (row["segment_uid"] + ".wav")
            row["segment_local_path"] = uc.to_project_relative(abs_path, config.project_root)
    uc.write_and_verify_segments(segments, context)

    # 3) Fingerprint eligible retained segment WAVs. Persist only the new batch.
    fingerprint_started = time.perf_counter()

    def fingerprint_row(row):
        return compute_fingerprint(
            uc.resolve_project_path(row["segment_local_path"], config.project_root), config.overlap,
        )

    segment_fingerprints_by_uid = uc.fingerprint_retained_u_segments(
        segments, context, fingerprint_row, cached_fps,
    )
    segment_fingerprint_seconds = time.perf_counter() - fingerprint_started

    # 4) Fingerprint protected references from pinned Parquet, one shard at a
    # time. Completed fingerprint shards are reused. Temp WAVs are not kept.
    # WAVs under 5.1s are audited as SHORT_NOT_PERCEPTUALLY_ELIGIBLE and are not fingerprinted.
    reference_started = time.perf_counter()
    uc.generate_protected_fingerprints(context, config.protected_reference_resolver)
    reference_fingerprint_seconds = time.perf_counter() - reference_started

    # 5) Exact protected PCM identity for every materialized reference, then
    # perceptual U-U dedup and fingerprinted protected shards only.
    context.completion.references_resolved = uc.references_cover_splits(context.protected_entries)
    overlap_ready = bool(config.overlap_config_frozen)
    segment_coverage = uc._fingerprint_coverage_complete(segments, segment_fingerprints_by_uid, config) if overlap_ready else False
    context.completion.segment_fingerprint_coverage_complete = segment_coverage
    evidence = list(uc.apply_protected_exact_identity(segments, context))
    if overlap_ready and segment_coverage:
        evidence.extend(uc.perceptual_deduplicate(segments, segment_fingerprints_by_uid, config))
        context.completion.u_u_protection_complete = True
        index, build_seconds = uc.build_u_candidate_index(segments, segment_fingerprints_by_uid, config)
        bench = {
            "index_build_elapsed_seconds": round(build_seconds, 6),
            "index_entry_count": int(index.entry_count),
            "n_references_fingerprinted": 0,
        }
        for batch in uc.iter_protected_fingerprint_shards(context):
            evidence.extend(uc.match_protected_batch(segments, index, batch, config))
            bench["n_references_fingerprinted"] += len(batch)
        bench["fingerprinting_elapsed_seconds"] = round(segment_fingerprint_seconds + reference_fingerprint_seconds, 6)
        provenance = dict(context.reference_provenance)
        provenance["protected_index_benchmark"] = bench
        context.reference_provenance = provenance
        context.completion.reference_fingerprint_coverage_complete = uc.protected_fingerprint_coverage_complete(context)
        context.completion.protected_overlap_check_complete = bool(
            context.completion.reference_fingerprint_coverage_complete
            and uc.protected_exact_identity_complete(context)
        )
    else:
        context.completion.u_u_protection_complete = False
        context.completion.reference_fingerprint_coverage_complete = False
        context.completion.protected_overlap_check_complete = False
    context.match_evidence = evidence

    # 6) Fail-closed validation, then compact protected shards into the final parquet.
    uc.validate_u_clean(segments, context)
    written = uc.write_u_clean_artifacts(
        segments, context,
        segment_fingerprints_by_uid=segment_fingerprints_by_uid,
        reference_fingerprints=None,
    )
    summary = uc.build_summary(segments, context)
    return summary, written


if RUN_FULL_PIPELINE:
    summary, written = run_pipeline(config)
    print(summary["status"])
    print("retained segments:", summary["n_segments_retained"], "| hours:", summary["retained_audio_hours"])
else:
    print("RUN_FULL_PIPELINE is False: skipping the real NB11 pipeline.")
    print("full NB11 requires: webrtcvad, fpcalc, pyarrow")
    print("Segmentation and overlap are frozen. similarity_threshold=0.775236, min_overlap_items=20.")


## 3. Offline synthetic checks

No network, no webrtcvad, no fpcalc. Injects a synthetic speech mask and synthetic fingerprints to
exercise the pure logic (UID determinism, no-overlap, exact dedup, perceptual dedup, protected overlap,
fail-closed validation). Mirrors `tests/test_rq2_u_clean.py`.

In [ ]:
import numpy as np
from dataclasses import replace
from src.rq2_segmentation import (
    plan_segments, segment_uid, segmentation_contract_sha256,
    assert_no_overlap, compute_frame_energies,
)
from src.rq2_audio_fingerprint import compare_fingerprints, OverlapConfig


def _sine_pcm(seconds, freq=220.0, sr=16000, amp=0.3):
    t = np.arange(int(seconds * sr)) / float(sr)
    return (np.sin(2 * np.pi * freq * t) * amp * 32767).astype("<i2")


def _src(sid):
    return {
        "source_id": sid,
        "article_url": "https://vov4.vov.vn/bahnar/x/" + sid + ".vov4",
        "article_date": "2026-02-01",
        "source_section": "kotong-ang-lom-topol-thoi-su-xa-hoi",
        "content_class": "news",
        "phase": "A",
        "media_url_sha256": "f" * 64,
        "wav_local_path": "artifacts/rq2/vov4_full/wav16k/" + sid + ".wav",
        "wav_sha256": "0" * 64, "pcm16_sha256": "0" * 64, "duration_seconds": "600.0",
        "source_level_status": "ELIGIBLE_SOURCE", "technical_status": "PASS",
    }


def run_synthetic_checks():
    seg_cfg = SegmentationConfig()
    frame_len = seg_cfg.frame_len_samples

    # UID determinism
    sha = segmentation_contract_sha256(seg_cfg)
    assert segment_uid("S", "a" * 64, 0, 480, sha) == segment_uid("S", "a" * 64, 0, 480, sha)
    assert segment_uid("S", "a" * 64, 0, 481, sha) != segment_uid("S", "a" * 64, 0, 480, sha)

    # 100s continuous speech splits with no overlap, each <= max
    n_frames = seg_cfg.seconds_to_frames(100.0)
    flags = np.ones(n_frames, dtype=bool)
    energies = np.full(n_frames, 0.5, dtype=float)
    plan = plan_segments(flags, energies, n_frames * frame_len, seg_cfg)
    assert len(plan.segments) >= 3
    assert_no_overlap(plan.segments)
    assert all(s.duration_seconds(seg_cfg.sample_rate) <= seg_cfg.max_segment_seconds + 1e-9 for s in plan.segments)

    # lowest-energy fallback picks the deep minimum near the target
    from src.rq2_segmentation import _choose_cut
    n = seg_cfg.seconds_to_frames(60.0)
    f2 = np.ones(n, dtype=bool)
    e2 = np.full(n, 0.9, dtype=float)
    low = seg_cfg.seconds_to_frames(24.0)
    e2[low] = 0.0
    assert _choose_cut(0, n, f2, e2, seg_cfg) == low

    review_cfg = uc.build_config(
        project_root=PROJECT_ROOT, segmentation=replace(seg_cfg, frozen=True),
        overlap=OverlapConfig(frozen=True, shingle_k=4, min_shared_shingles=1, min_overlap_items=4, similarity_threshold=0.9),
    )

    # exact dedup deterministic owner + canonical_segment_uid
    pcm = _sine_pcm(6.0)
    fr = len(pcm) // frame_len
    a = uc.build_segments_for_source(_src("VOV4_A"), pcm, np.ones(fr, bool), review_cfg)
    b = uc.build_segments_for_source(_src("VOV4_B"), pcm, np.ones(fr, bool), review_cfg)
    segments = a + b
    uc.exact_deduplicate(segments)
    assert len(uc.retained_segments(segments)) == 1
    owner_uid = uc.retained_segments(segments)[0]["segment_uid"]
    assert all(r["canonical_segment_uid"] == owner_uid for r in segments)

    # perceptual dedup + evidence (synthetic fingerprints)
    ctx = uc.UCleanContext(config=review_cfg)
    segs2 = []
    for sid, f in (("VOV4_C", 200.0), ("VOV4_D", 440.0)):
        p = _sine_pcm(6.0, freq=f); n2 = len(p) // frame_len
        segs2 += uc.build_segments_for_source(_src(sid), p, np.ones(n2, bool), review_cfg)
    rng = np.random.default_rng(0)
    fp = rng.integers(0, 2 ** 32, size=80, dtype=np.uint64).astype("<u4").tolist()
    uids = [s["segment_uid"] for s in segs2]
    uc.protect_and_deduplicate(segs2, ctx, segment_fingerprints_by_uid={uids[0]: fp, uids[1]: fp}, reference_fingerprints=[])
    assert len(uc.retained_segments(segs2)) == 1
    assert any(e.match_type == "u_u" for e in ctx.match_evidence)

    # fail closed on a retained protected overlap
    segs3 = []
    for sid, f in (("VOV4_E", 200.0), ("VOV4_F", 440.0)):
        p = _sine_pcm(6.0, freq=f); n3 = len(p) // frame_len
        segs3 += uc.build_segments_for_source(_src(sid), p, np.ones(n3, bool), review_cfg)
    segs3[0]["overlap_frozen_test"] = True
    try:
        uc.validate_u_clean(segs3, ctx)
        raise AssertionError("retained overlap did not fail closed")
    except RuntimeError:
        pass

    # success gate refuses without protection evidence, even in a full run
    full_cfg = uc.build_config(
        project_root=PROJECT_ROOT, run_full_pipeline=True,
        segmentation=replace(seg_cfg, frozen=True),
        overlap=OverlapConfig(frozen=True, shingle_k=4, min_shared_shingles=1, min_overlap_items=4, similarity_threshold=0.9),
    )
    ctx_no = uc.UCleanContext(config=full_cfg, nb10_status="SUCCESS_RQ2_VOV4_FULL_SOURCE_POOL")
    ctx_no.completion.references_resolved = True
    ctx_no.completion.nb10_locked = True
    ctx_no.completion.segments_written_verified = True
    assert uc.build_summary(segs2, ctx_no)["status"] == uc.FAIL_STATUS

    # reference loader forbids transcript columns
    try:
        uc.assert_no_forbidden_reference_columns(["record_uid", "text_bahnar"])
        raise AssertionError("forbidden column not rejected")
    except RuntimeError:
        pass

    # empty protected-reference fingerprints cannot complete protection
    ctx_empty_refs = uc.UCleanContext(config=review_cfg)
    ctx_empty_refs.protected_entries = []
    segs_empty = []
    for sid, f in (("VOV4_G", 300.0), ("VOV4_H", 500.0)):
        p = _sine_pcm(6.0, freq=f)
        n4 = len(p) // frame_len
        segs_empty += uc.build_segments_for_source(_src(sid), p, np.ones(n4, bool), review_cfg)
    fp_g = rng.integers(0, 2 ** 32, size=80, dtype=np.uint64).astype("<u4").tolist()
    fp_h = rng.integers(0, 2 ** 32, size=80, dtype=np.uint64).astype("<u4").tolist()
    uids_empty = [s["segment_uid"] for s in segs_empty]
    uc.protect_and_deduplicate(
        segs_empty, ctx_empty_refs,
        segment_fingerprints_by_uid={uids_empty[0]: fp_g, uids_empty[1]: fp_h},
        reference_fingerprints=[],
    )
    if ctx_empty_refs.completion.protected_overlap_check_complete:
        raise AssertionError("empty reference fingerprints were treated as protected")

    # media_url_sha256 is copied from NB10, not hashed from the article URL
    copied = uc.build_segments_for_source(
        _src("VOV4_MEDIA"), _sine_pcm(6.0), np.ones(len(_sine_pcm(6.0)) // frame_len, bool), review_cfg,
    )
    # _src has no media hash; supply one explicitly
    media_row = _src("VOV4_MEDIA2")
    media_row["media_url_sha256"] = "f" * 64
    copied = uc.build_segments_for_source(media_row, _sine_pcm(6.0), np.ones(len(_sine_pcm(6.0)) // frame_len, bool), review_cfg)
    if copied[0]["media_url_sha256"] != "f" * 64:
        raise AssertionError(copied[0]["media_url_sha256"])

    uc.scan_nb11_safety([
        PROJECT_ROOT / "src" / "rq2_segmentation.py",
        PROJECT_ROOT / "src" / "rq2_audio_fingerprint.py",
        PROJECT_ROOT / "src" / "rq2_u_clean.py",
        PROJECT_ROOT / "notebooks" / "11_RQ2_UReal_Segmentation_Dedup_Freeze_UClean.ipynb",
    ])

    incomplete = _src("VOV4_PROV")
    incomplete["article_url"] = ""
    try:
        uc.validate_source_contract([incomplete])
        raise AssertionError("missing provenance was accepted")
    except RuntimeError:
        pass
    uc.validate_source_contract([_src("VOV4_PROV_OK")])

    print("SYNTHETIC_CHECKS_PASS")




run_synthetic_checks()

# Compile every code cell of this notebook (self-check).
import json as _json
_nb = _json.loads(Path(PROJECT_ROOT / "notebooks" / "11_RQ2_UReal_Segmentation_Dedup_Freeze_UClean.ipynb").read_text(encoding="utf-8"))
_n = 0
for _cell in _nb.get("cells", []):
    if _cell.get("cell_type") == "code":
        compile("".join(_cell.get("source", [])), "nb11_cell", "exec")
        _n += 1
print("CODE_CELLS_COMPILED = " + str(_n) + "/" + str(_n))
